In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

# sample Employee info

data_employee = [
    Row(employee_id=1, name="John", city= 'New York'),
    Row(employee_id=2, name="Jane", city = 'Loss Angeles'),
    Row(employee_id=3, name="Bob", city = 'Chicago'),
    Row(employee_id=4, name="Alice", city = 'Houston'),
    Row(employee_id=5, name="Charlie", city = 'Miami')
]

# sample salary info

data_salary = [
    Row(employee_id=1, salary=50000,deprt = 'HR'),
    Row(employee_id=2, salary=60000,deprt = 'IT'),
    Row(employee_id=3, salary=70000,deprt = 'Engineering'),
    Row(employee_id=6, salary=80000,deprt = 'Marketing'),
    Row(employee_id=7, salary=90000,deprt = 'Sales'),
]

spark = SparkSession.getActiveSession()

df_employee = spark.createDataFrame(data_employee)
df_salary = spark.createDataFrame(data_salary)
# Register tempviews for SQL

# Employee info
df_employee.createOrReplaceTempView("employee")

# Salary info
df_salary.createOrReplaceTempView("salary")
#

display(df_employee)
display(df_salary)



### Broadcast Join Threshold

In [0]:
threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold") # default = 10MB
print(threshold)

#### Setting Broadcase join thresold

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1gb")
threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(threshold)

### Getting Physical Plan   

In [0]:
df_employee.join(df_salary,on = 'employee_id', how = 'inner').explain()

In [0]:
%sql
Explain
select * 
from employee
inner join salary
on employee.employee_id = salary.employee_id

### Hint for broadcast join

In [0]:
df = df_employee.join(df_salary.hint("broadcast"),on = 'employee_id', how = 'inner')
display(df)
df.explain()

In [0]:
%sql
SELECT /* + BROADCAST(salary) */
 employee.*,
 salary.salary,
 salary.deprt
FROM employee
INNER JOIN salary
ON employee.employee_id = salary.employee_id
